# Fine-tuning (notebook) vs. `new_fine_tune.py`

**Canonical pipeline (report + HFold):** run from the project root:

`python new_fine_tune.py`

That script (1) preserves pretrained last layers (no destructive reinit by default), (2) fine-tunes last **3** blocks with **regular** attention, (3) starts again from default weights, adds **sliding-window** on the last **3** layers, fine-tunes last 3, (4) runs **inference** `benchmark_three_modes` (full / sliding / HFold) on both checkpoints. See `CONFIG` at the top of that file.

The cells below are a **notebook mirror** of earlier helpers; they may not match the script’s CONFIG keys until you align them. For incremental debugging, use the same **Step**-style order with `torch_dtype=float32` on load, or call `import new_fine_tune as m; m.main()` after `m.CONFIG.update({...})` overrides.

In [1]:
!pip install torch datasets transformers python-dotenv tqdm

In [2]:


from dotenv import load_dotenv
import os
import math
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_cosine_schedule_with_warmup,
    DataCollatorForLanguageModeling,
)
from tqdm import tqdm
import random

load_dotenv()

# NOTING: THIS DOES NOT WORK FOR SCROLLS YET




False

## Config & `device`

In [3]:
CONFIG = {
    "cache_dir": "./data",
    "output_dir": "./checkpoints",

    "model_name": "EleutherAI/pythia-31m",   # or "gpt2"
    "dataset_name": "wikitext",              # "wikitext" or "scrolls_gov_report"
    "wikitext_config": "wikitext-103-raw-v1",
    "scrolls_config": "gov_report",         # used when dataset_name is scrolls_gov_report

    "quick_debug_epochs": False,             # if True, force 1 epoch per phase (smoke test)

    "max_length": 2048,
    "train_batch_size": 4,
    "eval_batch_size": 4,

    "phase1_epochs": 2,          # k
    "phase2_epochs": 3,          # m
    "last_k_layers": 2,          # start with 2; change to 1 if needed

    "lr_phase1": 1e-5,
    "lr_phase2": 5e-6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.03,

    "save_phase1_name": "last_k_finetuning",
    "save_phase2_name": "full_finetuning",

    "attention-format" : "regular",
    "seed" : 42,

    "sliding-length" : 256
}


device = "cuda" if torch.cuda.is_available() else "cpu"



## Data: `load_raw_dataset`, tokenize, `group_texts`, `build_dataloaders`

In [4]:
def load_raw_dataset(config):
    if config["dataset_name"] == "wikitext":
        dataset = load_dataset(
            "wikitext",
            config["wikitext_config"],
            cache_dir=config["cache_dir"],
        )
    elif config["dataset_name"] == "scrolls_gov_report":
        dataset = load_dataset(
            "tau/scrolls",
            config["scrolls_config"],
            cache_dir=config["cache_dir"],
            trust_remote_code=True,
        )
    else:
        raise ValueError(f"Unsupported dataset_name: {config['dataset_name']}")
    return dataset


def tokenize_function(examples, tokenizer, config):
    if config["dataset_name"] == "wikitext":
        texts = examples["text"]
    elif config["dataset_name"] == "scrolls_gov_report":
        texts = [
            f"Document:\n{inp}\n\nSummary:\n{out}"
            for inp, out in zip(examples["input"], examples["output"])
        ]
    else:
        raise ValueError(f"Unsupported dataset_name: {config['dataset_name']}")

    return tokenizer(
        texts,
        truncation=True,
        max_length=config["max_length"],
        padding=False,
    )


def group_texts(examples, block_size):
    concatenated = {}
    for k in examples.keys():
        concatenated[k] = sum(examples[k], [])

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {}
    for k, v in concatenated.items():
        result[k] = [v[i:i + block_size] for i in range(0, total_length, block_size)]

    result["labels"] = result["input_ids"].copy()
    return result


def build_dataloaders(tokenizer, config):
    raw_dataset = load_raw_dataset(config)

    if config["dataset_name"] == "wikitext":
        tokenized = raw_dataset.map(
            lambda examples: tokenizer(examples["text"], add_special_tokens=False),
            batched=True,
            remove_columns=raw_dataset["train"].column_names,
        )

        lm_dataset = tokenized.map(
            lambda examples: group_texts(examples, config["max_length"]),
            batched=True,
        )

        collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
        )

        train_loader = DataLoader(
            lm_dataset["train"],
            batch_size=config["train_batch_size"],
            shuffle=True,
            collate_fn=collator,
        )

        eval_split = "validation" if "validation" in lm_dataset else "test"
        eval_loader = DataLoader(
            lm_dataset[eval_split],
            batch_size=config["eval_batch_size"],
            shuffle=False,
            collate_fn=collator,
        )

    elif config["dataset_name"] == "scrolls_gov_report":
        tokenized = raw_dataset.map(
            lambda examples: tokenize_function(examples, tokenizer, config),
            batched=True,
            remove_columns=raw_dataset["train"].column_names,
        )

        collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False,
        )

        train_loader = DataLoader(
            tokenized["train"],
            batch_size=config["train_batch_size"],
            shuffle=True,
            collate_fn=collator,
        )

        eval_split = "validation" if "validation" in tokenized else "test"
        eval_loader = DataLoader(
            tokenized[eval_split],
            batch_size=config["eval_batch_size"],
            shuffle=False,
            collate_fn=collator,
        )

    else:
        raise ValueError(f"Unsupported dataset_name: {config['dataset_name']}")

    return train_loader, eval_loader



## `create_optimizer` & `create_scheduler`

In [5]:
def create_optimizer(model, lr, weight_decay):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    return AdamW(trainable_params, lr=lr, weight_decay=weight_decay)


def create_scheduler(optimizer, dataloader, num_epochs, warmup_ratio):
    total_steps = len(dataloader) * num_epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    return scheduler



## Reinitialize & freeze: `reinitialize_last_k_layers`, `freeze_all_but_last_k_layers`

In [6]:
def reinitialize_last_k_layers(model, k):
    modules_to_reset = []

    # Pythia / GPT-NeoX
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        modules_to_reset.extend(model.gpt_neox.layers[-k:])

        # add final layer norm and LM head
        if hasattr(model.gpt_neox, "final_layer_norm"):
            modules_to_reset.append(model.gpt_neox.final_layer_norm)
        # if hasattr(model, "embed_out"):
            # modules_to_reset.append(model.embed_out)

    # GPT-2
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        modules_to_reset.extend(model.transformer.h[-k:])

        # add final layer norm and LM head
        if hasattr(model.transformer, "ln_f"):
            modules_to_reset.append(model.transformer.ln_f)
        # if hasattr(model, "lm_head"):
            # modules_to_reset.append(model.lm_head)

    else:
        raise ValueError("Unsupported model architecture for reinitialization.")

    for target_module in modules_to_reset:
        for module in target_module.modules():
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()


def freeze_all_but_last_k_layers(model, k):
    for param in model.parameters():
        param.requires_grad = False

    # Pythia / GPT-NeoX
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        layers = model.gpt_neox.layers
        for layer in layers[-k:]:
            for param in layer.parameters():
                param.requires_grad = True

        if hasattr(model.gpt_neox, "final_layer_norm"):
            for param in model.gpt_neox.final_layer_norm.parameters():
                param.requires_grad = True

        if hasattr(model, "embed_out"):
            for param in model.embed_out.parameters():
                param.requires_grad = True

    # GPT-2
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = model.transformer.h
        for layer in layers[-k:]:
            for param in layer.parameters():
                param.requires_grad = True

        if hasattr(model.transformer, "ln_f"):
            for param in model.transformer.ln_f.parameters():
                param.requires_grad = True

        if hasattr(model, "lm_head"):
            for param in model.lm_head.parameters():
                param.requires_grad = True

    else:
        raise ValueError("Unsupported model architecture for freezing.")



## Sliding-window attention (optional)

In [7]:
class SlidingWindowAttentionWrapper(torch.nn.Module):
    """
    Wraps an existing HF attention module to enforce a sliding-window constraint
    on the attention mask during the forward pass.

    Handles asymmetric (non-square) masks so it works correctly during both
    training and auto-regressive generation with KV caching. Supports both
    additive float masks and boolean masks.
    """
    def __init__(self, original_attention, window_size):
        super().__init__()
        self.original_attention = original_attention
        self.window_size = window_size

    def forward(self, hidden_states, *args, **kwargs):
        # HF NeoX/older `GPTNeoXLayer` can pass the mask as the first *positional* arg to attention (not
        # in kwargs). Only checking `kwargs` then is a no-op. Some stacks pass a 2D user mask in kwargs
        # while the 4D causal mask is positional; only window the 4D mask.
        mask = kwargs.get("attention_mask")
        mask_pos = None
        if mask is not None and mask.dim() != 4:
            mask = None
        if mask is None and args:
            for i, a in enumerate(args):
                if torch.is_tensor(a) and a.dim() == 4:
                    mask = a
                    mask_pos = i
                    break
        if mask is not None:
            device = mask.device
            tgt_len = mask.shape[-2]
            src_len = mask.shape[-1]

            idx_tgt = torch.arange(src_len - tgt_len, src_len, device=device).unsqueeze(1)
            idx_src = torch.arange(src_len, device=device).unsqueeze(0)

            out_of_window = (idx_tgt - idx_src) >= self.window_size

            modified_mask = mask.clone()
            if modified_mask.dtype == torch.bool:
                modified_mask = modified_mask.masked_fill(out_of_window, False)
            else:
                min_val = torch.finfo(modified_mask.dtype).min
                modified_mask = modified_mask.masked_fill(out_of_window, min_val)

            if mask_pos is not None:
                args = list(args)
                args[mask_pos] = modified_mask
                args = tuple(args)
            else:
                kwargs = {**kwargs, "attention_mask": modified_mask}

        return self.original_attention(hidden_states, *args, **kwargs)


def apply_sliding_window_to_last_k_layers(model, k, window_size):
    """
    Replaces the attention module in the last k transformer layers with a
    SlidingWindowAttentionWrapper. Earlier layers retain full attention.
    Supports Pythia/GPT-NeoX and GPT-2.
    """
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        for layer in model.gpt_neox.layers[-k:]:
            layer.attention = SlidingWindowAttentionWrapper(layer.attention, window_size)
            print(f"Applied sliding-window attention (window={window_size}) to Pythia layer.")

    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        for layer in model.transformer.h[-k:]:
            layer.attn = SlidingWindowAttentionWrapper(layer.attn, window_size)
            print(f"Applied sliding-window attention (window={window_size}) to GPT-2 layer.")

    else:
        raise ValueError("Unsupported model architecture for sliding window.")



## `unfreeze_all_layers`, checkpoint, one epoch, `evaluate`

In [8]:
def unfreeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = True


def save_checkpoint(model, tokenizer, output_dir, checkpoint_name):
    save_path = os.path.join(output_dir, checkpoint_name)
    os.makedirs(save_path, exist_ok=True)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved checkpoint to {save_path}")


def train_one_epoch(model, dataloader, optimizer, scheduler):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss).all():
            raise RuntimeError(
                "Non-finite loss. Load the model with torch_dtype=torch.float32; "
                "Hub Pythia defaults to float16, which often diverges in LM training."
            )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(dataloader)
    perplexity = math.exp(avg_loss) if avg_loss < 25 else float("inf")
    return avg_loss, perplexity



## Phase runners: `run_last_k_finetuning`, `run_full_finetuning`

In [9]:
def run_last_k_finetuning(model, tokenizer, train_loader, eval_loader, config):
    print(f"\nStarting destructive last-{config['last_k_layers']}-layer fine-tuning...")

    # destroy the last k layers (+ LN and LM Head)
    reinitialize_last_k_layers(model, config["last_k_layers"])

    # freeze everything else
    freeze_all_but_last_k_layers(model, config["last_k_layers"])

    optimizer = create_optimizer(
        model,
        config["lr_phase1"],
        config["weight_decay"],
    )
    scheduler = create_scheduler(
        optimizer,
        train_loader,
        config["phase1_epochs"],
        config["warmup_ratio"],
    )

    for epoch in range(config["phase1_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        eval_loss, eval_ppl = evaluate(model, eval_loader)

        print(
            f"[Phase 1 | Epoch {epoch + 1}/{config['phase1_epochs']}] "
            f"train_loss={train_loss:.4f} | eval_loss={eval_loss:.4f} | eval_ppl={eval_ppl:.4f}"
        )

    save_checkpoint(model, tokenizer, config["output_dir"], config["save_phase1_name"])


def run_full_finetuning(model, tokenizer, train_loader, eval_loader, config):
    print("\nStarting full fine-tuning...")

    # keep current weights, just unfreeze all layers
    unfreeze_all_layers(model)

    # create a new optimizer for Phase 2 now that all layers are unfrozen
    optimizer = create_optimizer(
        model,
        config["lr_phase2"],
        config["weight_decay"],
    )

    scheduler = create_scheduler(
        optimizer,
        train_loader,
        config["phase2_epochs"],
        config["warmup_ratio"],
    )

    for epoch in range(config["phase2_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        eval_loss, eval_ppl = evaluate(model, eval_loader)

        print(
            f"[Phase 2 | Epoch {epoch + 1}/{config['phase2_epochs']}] "
            f"train_loss={train_loss:.4f} | eval_loss={eval_loss:.4f} | eval_ppl={eval_ppl:.4f}"
        )

    save_checkpoint(model, tokenizer, config["output_dir"], config["save_phase2_name"])



## Pipeline helpers (`setup_seeds_and_output_dir`, `load_tokenizer_and_model`, `run_full_pipeline`, `main`)

In [10]:
def get_training_config(config=None):
    """Return a config dict for training; optionally shorten epochs for quick_debug_epochs."""
    config = config or CONFIG
    c = dict(config)
    if c.get("quick_debug_epochs"):
        c["phase1_epochs"] = 1
        c["phase2_epochs"] = 1
    return c


def setup_seeds_and_output_dir(config):
    seed = config["seed"]
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.makedirs(config["output_dir"], exist_ok=True)


def load_tokenizer_and_model(config):
    print("Loading model/tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        config["model_name"],
        cache_dir=config["cache_dir"],
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Pythia's config.json uses torch_dtype=float16; training (esp. long ctx + reinit) in
    # fp16 without AMP often yields NaN — use float32 for stable finetuning.
    model = AutoModelForCausalLM.from_pretrained(
        config["model_name"],
        cache_dir=config["cache_dir"],
        torch_dtype=torch.float32,
    ).to(device)
    if config["attention-format"] == "sliding-window":
        apply_sliding_window_to_last_k_layers(
            model, config["last_k_layers"], config["sliding-length"]
        )
    return tokenizer, model


def run_full_pipeline(config):
    """Run setup → data → both phases. Sets notebook globals: tokenizer, model, train_loader, eval_loader."""
    global tokenizer, model, train_loader, eval_loader
    setup_seeds_and_output_dir(config)
    tokenizer, model = load_tokenizer_and_model(config)
    print("Building dataloaders...")
    train_loader, eval_loader = build_dataloaders(tokenizer, config)
    train_cfg = get_training_config(config)
    run_last_k_finetuning(model, tokenizer, train_loader, eval_loader, train_cfg)
    run_full_finetuning(model, tokenizer, train_loader, eval_loader, train_cfg)


def main():
    run_full_pipeline(CONFIG)



## Run training: incremental or full pipeline

Use the **Step** cells below to load and train in stages, or run **`run_full_pipeline(CONFIG)`** (same as **`main()`**) for the full two-phase run in one cell.

In [ ]:
# --- Step 1: seeds + output directory (fast) ---
setup_seeds_and_output_dir(CONFIG)

Loading model/tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Building dataloaders...


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wikitext-103-raw-v1' at data/wikitext/wikitext-103-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sat Apr 25 20:45:56 2026).


Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/1801350 [00:00<?, ? examples/s]

In [ ]:
# --- Step 2: tokenizer + model (download / load weights) ---
tokenizer, model = load_tokenizer_and_model(CONFIG)

In [ ]:
# --- Step 3: dataloaders (usually slow: tokenize + map) ---
train_loader, eval_loader = build_dataloaders(tokenizer, CONFIG)

In [ ]:
# --- Step 4: phase 1 — last-k layer fine-tuning ---
train_cfg = get_training_config()
run_last_k_finetuning(model, tokenizer, train_loader, eval_loader, train_cfg)

In [ ]:
# --- Step 5: phase 2 — full model fine-tuning ---
train_cfg = get_training_config()
run_full_finetuning(model, tokenizer, train_loader, eval_loader, train_cfg)

In [ ]:
# --- Optional: full pipeline in one cell (equivalent to Steps 1–5 + `main()`) ---
# run_full_pipeline(CONFIG)

In [ ]:
import os
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def test_post_main_checkpoints_from_disk():
    """Same as above, but recreates tokenizer + eval_loader (no in-memory main)."""
    tok = AutoTokenizer.from_pretrained(CONFIG["model_name"], cache_dir=CONFIG["cache_dir"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    _, el = build_dataloaders(tok, CONFIG)

    paths = {
        "Phase 1 (last-k)": os.path.join(CONFIG["output_dir"], CONFIG["save_phase1_name"]),
        "Phase 2 (full)": os.path.join(CONFIG["output_dir"], CONFIG["save_phase2_name"]),
    }
    for label, path in paths.items():
        if not os.path.isdir(path):
            print(f"MISSING: {label} -> {path}")
            continue
        m = AutoModelForCausalLM.from_pretrained(
            path,
            local_files_only=True,
            cache_dir=CONFIG["cache_dir"],
            torch_dtype=torch.float32,
        ).to(device)
        m.eval()
        eloss, eppl = evaluate(m, el)
        print(f"{label}\n  path={path}\n  eval_loss={eloss:.4f}  eval_ppl={eppl:.4f}\n")
        del m
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

test_post_main_checkpoints_from_disk()